# 05 · Evaluación offline de la primera ACT

La pregunta: **¿la política aprendió algo que sirva en episodios que no vio?**

Cómo se mide: en distintos momentos de un episodio grabado le damos a la
política la observación real (cámara y articulaciones) y le pedimos su próximo
bloque de 48 acciones. Comparamos contra lo que hizo el operador.

Dos advertencias antes de mirar números:

- Es *teacher forcing*: la política siempre ve la observación verdadera, nunca
  la consecuencia de sus propios errores. Un buen número acá es necesario pero
  no suficiente; que la tarea salga se mide en lazo cerrado (simulación o robot).
- Un error en radianes no dice nada solo. Por eso lo comparamos con dos
  políticas tontas: **quedarse quieto** (`hold`) y **repetir la última acción**
  (`last_action`). Si ACT no les gana, no aprendió nada útil.

Para tener una escala física: 0,05 rad son unos 3 grados. En un brazo de
~50 cm, 3 grados en el hombro mueven la mano unos 2,5 cm. Para agarrar una
prenda doblada es tolerable; para enhebrar una aguja, no.

Qué hay adentro de ACT y por qué predice bloques: `notebooks/learn/L3`.

In [ ]:
import torch  # noqa: F401  (antes que lerobot, ver notebook 01)
import json

import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.eval.offline import episode_frames, error_by_horizon, error_table, evaluate_episode
from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy.load import load_policy

plt.rcParams["figure.dpi"] = 110

Cargamos la política **con** su pre y postprocesador. Sin ellos la red recibe
números en otra escala y todo lo que sigue sería basura (nos pasó en el robot).

In [ ]:
loaded = load_policy(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model", device="cpu")
print("campos de config descartados:", loaded.dropped_config_fields)
print("bloque:", loaded.policy.config.chunk_size, "acciones")

ds = LeRobotDataset("local/pick_mameluco_act_v1", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1", video_backend="pyav")
split = json.loads((REPO_ROOT / "learning" / "act" / "split.json").read_text())
JOINTS = ds.meta.features["action"]["names"][0]
print(len(split["heldout_episode_indices"]), "episodios no vistos,", len(split["train_episode_indices"]), "de entrenamiento")

Corremos sobre los dos conjuntos. Los de entrenamiento no sirven para decir si
generaliza, pero la diferencia entre ambos mide cuánto memorizó.

In [ ]:
heldout = [p for e in split["heldout_episode_indices"] for p in evaluate_episode(loaded, ds, e)]
train = [p for e in split["train_episode_indices"] for p in evaluate_episode(loaded, ds, e)]
summary = {"no vistos": error_table(heldout, JOINTS).mean(), "entrenamiento": error_table(train, JOINTS).mean()}
import pandas as pd
pd.DataFrame(summary).T.round(4)

Lectura rápida:

- En episodios no vistos ACT le gana a las dos referencias.
- Sobre los de entrenamiento el error es varias veces menor: hay
  **sobreajuste**. Con 54 demostraciones es esperable; es una de las cosas a
  atacar cuando reentrenemos (más datos, más aumentación, menos pasos).

## El error según cuán lejos predice

Esta es la curva más informativa. El eje x es "cuántos pasos hacia el futuro".

In [ ]:
horizon = error_by_horizon(heldout)
ax = horizon.plot(figsize=(9, 3.5))
ax.set_xlabel("pasos hacia adelante (30 por segundo)"); ax.set_ylabel("error medio (rad)")
ax.set_title("episodios no vistos")
plt.tight_layout()
horizon.iloc[[0, 5, 11, 23, 47]].round(4)

Fijate el arranque: para el **primer paso**, repetir la última acción es casi
perfecto (el operador no cambia mucho en 33 ms) y ACT erra bastante más. ACT
recién le gana a esa referencia después de unos cuantos pasos, y de ahí en
adelante la diferencia crece a su favor.

¿Por qué importa? Si el robot ejecuta los 48 pasos y pide un bloque nuevo, el
primer paso del bloque nuevo puede "saltar" respecto del último del anterior.
En el robot eso se ve como tirones. Es exactamente el problema que el paper de
ACT ataca con *temporal ensembling* (promediar las predicciones superpuestas de
varios bloques), que este modelo tiene apagado. Anotado para probar.

## Qué articulaciones le cuestan

In [ ]:
per_joint = error_table(heldout, JOINTS).sort_values("policy")
ax = per_joint.plot.barh(figsize=(8, 8))
ax.set_xlabel("error medio (rad)")
plt.tight_layout()

## Mirarlo, no sólo medirlo

Un episodio no visto, una articulación: en gris lo que hizo el operador, en
color cada bloque que predijo ACT desde su punto de partida.

In [ ]:
episode = split["heldout_episode_indices"][1]
joint = JOINTS.index("kRightElbow")
dense = evaluate_episode(loaded, ds, episode, stride=24)
truth = np.stack(episode_frames(ds, episode)["action"].to_numpy())

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(truth[:, joint], color="0.6", lw=3, label="operador")
for p in dense:
    ax.plot(range(p.start, p.start + len(p.predicted)), p.predicted[:, joint], lw=1)
ax.set_title(f"episodio {episode}, {JOINTS[joint]}"); ax.set_xlabel("cuadro"); ax.legend()
plt.tight_layout()

Cosas para buscar en este gráfico: si los bloques arrancan pegados a la curva
gris (si no, ahí está el salto del primer paso) y si se abren hacia el final
(el error crece con el horizonte).

## Lo que nos llevamos

- ACT aprendió algo real: le gana a las referencias en datos no vistos.
- Sobreajusta bastante.
- El primer paso de cada bloque es su punto débil. Candidatos: temporal
  ensembling o ejecutar menos pasos por bloque.

Todo esto es condición necesaria. La prueba de verdad es en lazo cerrado.